# Architecture Effect Analysis: 1-Stage vs 2-Stage H-Net

**Research Questions (Section E):**
- **E**: How does 1-stage vs 2-stage architecture affect tokenization under identical conditions?
- **E.1**: How do the two chunking levels (Stage 0 and Stage 1) relate in 2-stage models?
- **E.2**: What chemical patterns does each chunking stage learn?
- **E.3**: Does 2-stage benefit polymers more than molecules?
- **E.4**: Does 2-stage + concatenation show synergy?

## Model Comparisons

We compare models trained under identical conditions except for architecture:

### PI1M (Polymer) - Concatenated, 5 epochs:
- **1-stage**: `run_large_20251111_181836`
- **2-stage**: *(pending training)*

### MOSES (Molecular) - Concatenated, 5 epochs:
- **1-stage**: `run_large_20251112_071557`
- **2-stage**: *(pending training)*

## Analysis Goals
1. Compare Stage 0 tokenization between 1-stage and 2-stage models
2. Analyze Stage 1 super-chunking patterns (2-stage only)
3. Investigate chemical interpretability of each stage
4. Compare architecture effect on polymers vs molecules
5. Visualize hierarchical chunking structure


In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import pickle

# Setup plotting style
sns.set_style("whitegrid")
sns.set_context("talk")
colors = sns.color_palette("mako", 10)
plt.rcParams['figure.figsize'] = (14, 8)

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

# Import analysis utilities
from analysis.utils.statistics import TokenStatistics, compare_token_distributions, compute_kl_divergence
from analysis.utils.inference import get_model_info, get_architecture_type, get_stage_statistics

print("Imports successful!")
print(f"Project root: {project_root}")


## 1. Define Model Paths

Define the checkpoint directories for all models to compare.
**⚠️ Update the 2-stage paths after training completes!**


In [ ]:
# Model checkpoint paths
# ----------------------
# TODO: Update 2-stage paths once training is complete!

MODELS = {
    # 1-stage models (existing)
    'PI1M_1stage_concat_5ep': {
        'checkpoint_dir': project_root / 'checkpoints' / 'run_large_20251111_181836',
        'dataset': 'PI1M',
        'architecture': '1-stage',
        'concatenation': True,
        'epochs': 5,
    },
    'MOSES_1stage_concat_5ep': {
        'checkpoint_dir': project_root / 'checkpoints' / 'run_large_20251112_071557',
        'dataset': 'MOSES',
        'architecture': '1-stage',
        'concatenation': True,
        'epochs': 5,
    },
    
    # 2-stage models (pending - UPDATE PATHS AFTER TRAINING!)
    'PI1M_2stage_concat_5ep': {
        'checkpoint_dir': None,  # TODO: Update after training completes
        'dataset': 'PI1M',
        'architecture': '2-stage',
        'concatenation': True,
        'epochs': 5,
    },
    'MOSES_2stage_concat_5ep': {
        'checkpoint_dir': None,  # TODO: Update after training completes
        'dataset': 'MOSES',
        'architecture': '2-stage',
        'concatenation': True,
        'epochs': 5,
    },
}

# Display model status
print("Models for comparison:")
print("=" * 80)
for name, info in MODELS.items():
    if info['checkpoint_dir'] and Path(info['checkpoint_dir']).exists():
        status = "✓ Ready"
    else:
        status = "⏳ Pending"
    print(f"  {name}: {info['architecture']} | {info['dataset']} | {status}")


## 2. Load Tokenization Results

Load pre-computed tokenization results for each model.


In [ ]:
# Load tokenization results and statistics
results_dir = project_root / 'analysis' / 'data' / 'hnet_results'
stats_dir = project_root / 'analysis' / 'data' / 'statistics'

loaded_results = {}
loaded_stats = {}

for model_name, model_info in MODELS.items():
    # Check for existing results (use naming convention from data generation)
    # Map model names to existing file patterns
    name_mapping = {
        'PI1M_1stage_concat_5ep': 'PI1M_concat_5epoch',
        'MOSES_1stage_concat_5ep': 'MOSES_concat_5epoch',
        'PI1M_2stage_concat_5ep': 'PI1M_2stage_concat_5epoch',  # Future
        'MOSES_2stage_concat_5ep': 'MOSES_2stage_concat_5epoch',  # Future
    }
    
    file_name = name_mapping.get(model_name, model_name)
    result_file = results_dir / f"{file_name}_tokenization.pkl"
    stats_file = stats_dir / f"{file_name}_stats.json"
    
    if result_file.exists():
        print(f"Loading results for {model_name}...")
        with open(result_file, 'rb') as f:
            loaded_results[model_name] = pickle.load(f)
        print(f"  ✓ Loaded {len(loaded_results[model_name])} samples")
    else:
        print(f"  ⏳ Results not yet available: {model_name}")
        loaded_results[model_name] = None
    
    if stats_file.exists():
        loaded_stats[model_name] = TokenStatistics.load(str(stats_file))
    else:
        loaded_stats[model_name] = None

print("\nLoading complete!")


---

## Question E: Architecture Effect (1-stage vs 2-stage)

Compare Stage 0 tokenization between 1-stage and 2-stage models under identical conditions.


In [ ]:
def compare_architectures(stats_1stage, stats_2stage, dataset_name):
    """Compare 1-stage vs 2-stage architecture for the same dataset."""
    if stats_1stage is None or stats_2stage is None:
        print(f"⏳ Cannot compare {dataset_name} - waiting for 2-stage results")
        return None
    
    comparison = compare_token_distributions(
        stats_1stage, stats_2stage,
        label1=f"{dataset_name} 1-stage", label2=f"{dataset_name} 2-stage"
    )
    comparison['kl_divergence'] = compute_kl_divergence(stats_1stage, stats_2stage)
    return comparison

def analyze_chunking_hierarchy(results, model_name):
    """Analyze hierarchical chunking structure of 2-stage models."""
    if results is None or results[0].get('num_stages', 1) < 2:
        print(f"⏳ {model_name}: Not available or not 2-stage")
        return None
    
    stage0_sizes = [len(t) for r in results for t in r.get('tokens', [])]
    super_chunk_sizes = [len(sc) for r in results for sc in r.get('super_chunks', [])]
    chunks_per_sc = [c for r in results for c in r.get('chunks_per_super_chunk', [])]
    
    return {
        'stage0_chunk_size_mean': np.mean(stage0_sizes) if stage0_sizes else 0,
        'super_chunk_size_mean': np.mean(super_chunk_sizes) if super_chunk_sizes else 0,
        'chunks_per_super_chunk_mean': np.mean(chunks_per_sc) if chunks_per_sc else 0,
    }

print("Analysis functions defined ✓")


In [ ]:
# Run architecture comparisons (will show pending message until 2-stage training completes)

print("=" * 70)
print("ARCHITECTURE COMPARISON: 1-STAGE vs 2-STAGE")
print("=" * 70)

# PI1M comparison
pi1m_arch_comparison = compare_architectures(
    loaded_stats.get('PI1M_1stage_concat_5ep'),
    loaded_stats.get('PI1M_2stage_concat_5ep'),
    'PI1M (Polymer)'
)

# MOSES comparison  
moses_arch_comparison = compare_architectures(
    loaded_stats.get('MOSES_1stage_concat_5ep'),
    loaded_stats.get('MOSES_2stage_concat_5ep'),
    'MOSES (Molecular)'
)

# Hierarchy analysis (2-stage only)
print("\n" + "=" * 70)
print("CHUNKING HIERARCHY ANALYSIS (2-Stage Models)")
print("=" * 70)

pi1m_hierarchy = analyze_chunking_hierarchy(
    loaded_results.get('PI1M_2stage_concat_5ep'), 'PI1M_2stage'
)
moses_hierarchy = analyze_chunking_hierarchy(
    loaded_results.get('MOSES_2stage_concat_5ep'), 'MOSES_2stage'
)


---

## Summary: Key Findings (To be completed after training)

| Question | PI1M (Polymer) | MOSES (Molecular) | Interpretation |
|----------|---------------|-------------------|----------------|
| **E: Architecture Effect** | ⏳ Pending | ⏳ Pending | How different are 1-stage vs 2-stage tokenizations? |
| **E.1: Hierarchy Structure** | ⏳ Pending | ⏳ Pending | What are typical Stage 0/Stage 1 chunk sizes? |
| **E.2: Chemistry Interpretation** | ⏳ Pending | ⏳ Pending | What chemical patterns does each stage learn? |
| **E.3: Dataset Interaction** | ⏳ Pending | ⏳ Pending | Does 2-stage benefit polymers more? |
| **E.4: Concat Synergy** | ⏳ Pending | ⏳ Pending | Does 2-stage amplify concat benefits? |

### Next Steps
1. Complete 2-stage training runs
2. Update `MODELS` dictionary with actual checkpoint paths
3. Re-run this notebook to generate complete analysis
4. Update FINAL_REPORT.md with Section E results
